In [11]:
"""1.This cell contains the code loading,checking and reshaping the dataset for further processing""" 

import torch 
import numpy as np

#Loading Nmpy Arrays from the dataset
raw = np.load("flowers_data.npy")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Fallback to CPU if GPU is not available
if(device.type == "cuda"):
    print("GPU is available")
    print("Converting Arrays to Tensors . . . .")
    #Converting the numpy arrays to torch tensors for faster computing
    converted_x_label = torch.from_numpy(raw['X']).float() #W have converted to float because its necessary for further calculation accuracy
    converted_y_label = torch.from_numpy(raw['Y']).float()
print(converted_x_label.shape,converted_x_label.device)
print(converted_y_label.shape,converted_y_label.device)
print("Tensors Created!")

#So the shape becomes (N, H, W, C) where N is the number of samples, H is the height of the image, W is the width of the image and C is the number of channels. But the model we are using expects the input shape to be (N, C, H, W). So we need to permute the dimensions of the tensor to match the input shape of the model.
permuted = torch.permute(converted_x_label, (0, 3, 1, 2)) #Permuting the dimensions of the tensor to match the input shape of the model


GPU is available
Converting Arrays to Tensors . . . .
torch.Size([11197, 150, 150, 3]) cpu
torch.Size([11197]) cpu
Tensors Created!


In [12]:
"""This cell is used to shuffle the X and Y labels so that the model is 
actually learning and not memorizing the whole dataset

"""
indices = torch.randperm(len(permuted))
X_shuffled = permuted[indices] / 255.0
Y_shuffled = converted_y_label[indices].long().squeeze()


In [ ]:
#This cell splits the shuffled data into Training ans Test sets
#80% of the data is used for training and 20% for testing
train_split = int(0.8 * len(X_shuffled))
X_train = X_shuffled[:train_split]
Y_train = Y_shuffled[:train_split]
X_test = X_shuffled[train_split:]
Y_test = Y_shuffled[train_split:] 
print(len(X_train),len(Y_train),len(X_test),len(Y_test))



8957 8957 2240 2240


In [ ]:
from torch.utils.data import DataLoader
batch__size = 64


In [15]:
#Moving the tensors to the device (GPU or CPU) for faster computing
move_x_train = X_train.to(device) 
move_y_label = Y_train.to(device)
print(f"Moved X and Y label of training data to {device} successfully") 
move_x_test = X_test.to(device)
move_y_test = Y_test.to(device)
print(f"Moved X and Y label of testing data to {device} successfully")

Moved X and Y label of training data to cuda successfully
Moved X and Y label of testing data to cuda successfully


In [16]:
from torch import nn

class FlowerClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        #Neural Network Initilization

        self.layer_1 = nn.Linear(in_features = 67500,out_features =128,bias=True)
        self.relu_1 = nn.ReLU()
        self.layer_2 = nn.Linear(in_features=128,out_features=64,bias=True)
        self.relu_2 = nn.ReLU()
        self.layer_3 = nn.Linear(in_features=64,out_features=32,bias=True)
        self.relu_3 = nn.ReLU()

        #Output Layer
        self.out = nn.Linear(in_features = 32, out_features=7)

    def forward1(self,x):
        self.flatten = nn.Flatten()
        x = self.flatten(x)
        x = self.relu_1(self.layer_1(x))
        x = self.relu_2(self.layer_2(x))
        x = self.relu_3(self.layer_3(x))
        x = self.out(x)
        return x


In [ ]:
#